In [1]:
import numpy as np
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras_preprocessing import image
from keras.applications.vgg16 import VGG16, preprocess_input
from keras.layers import Dense, Flatten, Dropout

In [2]:
# Load the VGG16 model without the top layer
# This allows us to use the convolutional base for feature extraction
conv_base = VGG16(
    weights='imagenet',
    include_top=False,  # Exclude the fully connected layers at the top
    input_shape=(150, 150, 3)
)

In [3]:
conv_base.summary()

Model: "vgg16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 150, 150, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 150, 150, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 150, 150, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 75, 75, 64)        0         
                                                                 
 block2_conv1 (Conv2D)       (None, 75, 75, 128)       73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 75, 75, 128)       147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 37, 37, 128)       0     

### Fine-Tuning the Conv Base
- we can fine tune a few conv layers to train them specific to our data
- Freezing all the Conv Layers except the last one

In [4]:
set_trainable = False  # Flag to control which layers are trainable
for layer in conv_base.layers:  # Freeze all layers except the last 4
    if layer.name == 'block5_conv1':
        set_trainable = True

    if set_trainable:
        layer.trainable = True
    else:
        layer.trainable = False

for layer in conv_base.layers:
    print(layer.name, layer.trainable)

input_1 False
block1_conv1 False
block1_conv2 False
block1_pool False
block2_conv1 False
block2_conv2 False
block2_pool False
block3_conv1 False
block3_conv2 False
block3_conv3 False
block3_pool False
block4_conv1 False
block4_conv2 False
block4_conv3 False
block4_pool False
block5_conv1 True
block5_conv2 True
block5_conv3 True
block5_pool True


### add fine-tuned conv layers and add the Classification FC Layer

In [5]:
clf_model = Sequential(
    [
        conv_base,  # Add the convolutional base
        Flatten(),  # Flatten the output of the convolutional base
        Dense(256, activation='relu'),  # Fully connected layer with 256 units
        Dropout(0.5),  # Dropout layer to reduce overfitting
        # Output layer for binary classification
        Dense(1, activation='sigmoid')
    ]
)
# Freeze the convolutional base to prevent its weights from being updated during training

In [6]:
clf_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg16 (Functional)          (None, 4, 4, 512)         14714688  
                                                                 
 flatten (Flatten)           (None, 8192)              0         
                                                                 
 dense (Dense)               (None, 256)               2097408   
                                                                 
 dropout (Dropout)           (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 1)                 257       
                                                                 
Total params: 16,812,353
Trainable params: 9,177,089
Non-trainable params: 7,635,264
_________________________________________________________________


In [7]:
train_ds = keras.utils.image_dataset_from_directory(
    directory='data/train',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(150, 150),
)
test_ds = keras.utils.image_dataset_from_directory(
    directory='data/test',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(150, 150),
)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.


In [8]:
#  Preprocess the images by scaling pixel values to [0, 1] range
# and applying the VGG16 preprocessing function
def preprocess(image, label):
    image = tensorflow.cast(image/255., tensorflow.float32)
    return image, label


train_ds = train_ds.map(preprocess)
test_ds = test_ds.map(preprocess)

In [ ]:
clf_model.compile(
    # Use Adam optimizer with a small learning rate
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',  # Use binary crossentropy for binary classification
    metrics=['accuracy']
)

In [11]:
history = clf_model.fit(train_ds,
                        epochs=10,
                        validation_data=test_ds,
                        verbose="auto",
                        use_multiprocessing=True,
                        workers=4
                        )

Epoch 1/10
625/625 [==============================] - 108s 171ms/step - loss: 0.1713 - accuracy: 0.9302 - val_loss: 0.1270 - val_accuracy: 0.9478
Epoch 2/10
625/625 [==============================] - 105s 168ms/step - loss: 0.0915 - accuracy: 0.9638 - val_loss: 0.1838 - val_accuracy: 0.9386
Epoch 3/10
625/625 [==============================] - 105s 168ms/step - loss: 0.0501 - accuracy: 0.9800 - val_loss: 0.1414 - val_accuracy: 0.9546
Epoch 4/10
625/625 [==============================] - 105s 169ms/step - loss: 0.0301 - accuracy: 0.9888 - val_loss: 0.1485 - val_accuracy: 0.9556
Epoch 5/10
625/625 [==============================] - 105s 168ms/step - loss: 0.0235 - accuracy: 0.9917 - val_loss: 0.1463 - val_accuracy: 0.9556
Epoch 6/10
625/625 [==============================] - 105s 168ms/step - loss: 0.0165 - accuracy: 0.9945 - val_loss: 0.2022 - val_accuracy: 0.9516
Epoch 7/10
625/625 [==============================] - 106s 170ms/step - loss: 0.0182 - accuracy: 0.9940 - val_loss: 0.1658 -

In [ ]:
history

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], color='red', label='train')
plt.plot(history.history['val_accuracy'], color='blue', label='validation')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['loss'], color='red', label='train')
plt.plot(history.history['val_loss'], color='blue', label='validation')
plt.legend()
plt.show()